# 00 Question contract

**Owner:** all three, 1:00–1:20. Nobody fits a model before this page is filled in.

**Inputs:** files dropped in `data/raw/`, and the questions on the Moodle brief.

**Outputs:** `outputs/question_contract.json`, a draft `outputs/column_roles.json` once a file exists, and `outputs/graph_gate.json` copied from the contract.

**Stop rule:** this notebook profiles and drafts roles. It does not clean, join, or fit.

Edit the contract in the next cell, then run the rest. Set `native_graph` to `true` or `false` and `graph_kind` to `coordinates`, `edge_list`, or `shared_id`. Leave `model_needed` null until you know a question actually needs a prediction.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import (
    RAW,
    PROCESSED,
    OUTPUTS,
    TABLES,
    ANALYSIS_PATH,
    FEATURES_PATH,
    MODEL_MATRIX_PATH,
    ROLES_PATH,
    CONTRACT_PATH,
    GATE_PATH,
    FEATURE_LOG_PATH,
    ALIAS_PATH,
    ANSWERS_PATH,
    COMPARISON_PATH,
    BUNDLE_PATH,
    ensure_dirs,
)

ensure_dirs()
print("root", ROOT)


In [ ]:
import json

contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
print(json.dumps(contract, indent=2))
print("Edit outputs/question_contract.json, save it, then run the following cells again.")


In [ ]:
import json

from src import io as sio
from src.roles import suggest_roles, write_roles

files = sio.list_raw_files(RAW)
print(f"{len(files)} file(s) in data/raw")
if not files:
    print("No Moodle files yet. Drop them in data/raw and run this cell again.")
else:
    widest = None
    widest_cols = -1
    for path in files:
        frame = sio.load_table(path)
        profile_path = sio.write_profile(frame, path.stem)
        print(path.name, frame.shape, "->", profile_path.name)
        if frame.shape[1] > widest_cols:
            widest = frame
            widest_name = path.name
            widest_cols = frame.shape[1]
    if not ROLES_PATH.exists():
        draft = suggest_roles(widest)
        draft["source_file"] = widest_name
        write_roles(draft, ROLES_PATH)
        print("Draft roles written from", widest_name, "->", ROLES_PATH)
        print("Edit the draft. Name heuristics are only a start.")
    else:
        print("Roles already exist and were not overwritten:", ROLES_PATH)


In [ ]:
import json

from src.graph_gate import write_gate

contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
decision = contract.get("native_graph")
reason = contract.get("graph_reason") or "see the question contract"
kind = contract.get("graph_kind") or None
if decision is True or decision is False:
    write_gate(bool(decision), reason, kind=kind, evidence=contract.get("entity_risk") or "")
    print("Graph gate set from the contract:", decision, kind)
else:
    write_gate(None, "not decided", evidence="Set native_graph in outputs/question_contract.json")
    print("Graph gate is still undecided. Set native_graph true or false before 1:20.")
print("Questions:")
for question in contract.get("questions", []):
    print(question.get("id"), question.get("owner"), question.get("text") or "(empty)")
